[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/03_conditioning_and_condition_numbers/exercises.ipynb)

# Exercises — Topic 03: Conditioning and Condition Numbers

20 fully solved problems in 4 levels. Attempt each before opening its solution cell.

## Level 0 — Concept Check

### Problem L0.1: Conditioning versus stability

Classify each statement as true or false, with a one-line justification.

(a) "A large condition number means my code has a bug."
(b) "A backward stable algorithm always returns an accurate answer."
(c) "The condition number depends on the working precision."
(d) "Doubling the precision halves the condition number."


**Solution**

**(a) False.** $\kappa$ is defined by the map $f$ and the point $x$ alone — no algorithm, no code, no arithmetic appears in Definition 2.1. A huge $\kappa$ says the *question* is sensitive.

**(b) False.** Backward stability gives forward error $O(\kappa u)$. On an ill-conditioned problem a perfect algorithm still returns garbage: it is the exact answer to a nearby question, and nearby questions have far-apart answers.

**(c) False.** Same reason as (a): $\kappa$ is a property of the mathematical map. Precision determines the *size of the perturbation* $u$ that $\kappa$ amplifies, not $\kappa$ itself.

**(d) False.** Doubling precision (say $u : 10^{-16} \to 10^{-32}$) leaves $\kappa$ untouched but shrinks the product $\kappa u$, buying back digits. It moves the *input error*, not the lever arm.

$$
\boxed{\text{forward error} \approx \kappa \cdot u : \quad \kappa \text{ is the problem, } u \text{ is the precision, and only } u \text{ is yours to choose}}
$$

*Key takeaway*: never debug a conditioning problem, and never regularize an instability — diagnose which one you have first.

### Problem L0.2: Scalar condition numbers

Compute $\kappa_f(x) = \vert x f'(x)/f(x) \vert$ for (a) $f(x) = \sqrt{x}$, (b) $f(x) = e^{x}$ at $x = 40$, (c) $f(x) = \log x$ at $x = 1 + 10^{-10}$, (d) $f(x) = \tan x$ at $x = \pi/2 - 10^{-8}$. Interpret each.

**Solution**

**(a)** $f' = \tfrac{1}{2}x^{-1/2}$, so

$$
\kappa = \left\vert \frac{x \cdot \tfrac{1}{2}x^{-1/2}}{x^{1/2}} \right\vert = \frac{1}{2}
$$

Square root is *better* than perfectly conditioned: it halves relative error. (Squaring, conversely, has $\kappa = 2$.)

**(b)** $\kappa = \vert x e^{x}/e^{x} \vert = \vert x \vert = 40$. Exponentials are ill-conditioned for large arguments: a relative input error $10^{-16}$ in $x = 40$ becomes $4 \times 10^{-15}$ in $e^{x}$ — still fine, but at $x = 10^{15}$ the output has no correct digits.

**(c)** $\kappa = \vert x \cdot (1/x) / \log x \vert = 1/\vert \log x \vert$. With $\log(1 + 10^{-10}) \approx 10^{-10}$:

$$
\kappa \approx 10^{10}
$$

This is the analytic reason `log1p` exists (Topic 02): $x \mapsto \log x$ near $x = 1$ is intrinsically ill-conditioned, so one must change the *input parameterization* to the offset $x - 1$, for which $t \mapsto \mathrm{log1p}(t)$ has $\kappa = \vert t/((1+t)\log(1+t)) \vert \approx 1$.

**(d)** $f' = \sec^{2}x$, so $\kappa = \vert x \sec^{2}x / \tan x \vert = \vert x / (\sin x \cos x) \vert = \vert 2x/\sin 2x \vert$. At $x = \pi/2 - \epsilon$ with $\epsilon = 10^{-8}$: $\cos x \approx \epsilon$, $\sin x \approx 1$, so

$$
\kappa \approx \frac{\pi/2}{10^{-8}} \approx 1.6 \times 10^{8}
$$

$$
\boxed{\kappa_{\sqrt{\cdot}} = \tfrac{1}{2}, \quad \kappa_{\exp}(40) = 40, \quad \kappa_{\log}(1 + 10^{-10}) \approx 10^{10}, \quad \kappa_{\tan} \approx 1.6 \times 10^{8}}
$$

*Key takeaway*: $\kappa_f$ blows up wherever $f$ is near a zero (denominator small) or a pole (derivative large) — inspect those two places before trusting a formula.

### Problem L0.3: The determinant is not a condition number

Give two matrices showing that $\det$ carries no information about conditioning: (a) tiny determinant, perfect conditioning; (b) determinant $1$, terrible conditioning.

**Solution**

**(a)** $A = 10^{-5} I_{10}$. Then $\det A = 10^{-50}$, numerically indistinguishable from zero in many contexts, yet all singular values equal $10^{-5}$:

$$
\kappa_2(A) = \frac{10^{-5}}{10^{-5}} = 1
$$

Solving $Ax = b$ is a division of every component by $10^{-5}$ — exact to a rounding.

**(b)** $B = \mathrm{diag}(10^{8}, 10^{-8})$. Then $\det B = 1$ exactly, but

$$
\kappa_2(B) = \frac{10^{8}}{10^{-8}} = 10^{16}
$$

— the worst possible conditioning in binary64.

The structural reason: $\det A = \prod_i \sigma_i$ (up to sign) is a *product* of scales and rescales as $\det(cA) = c^{n}\det A$, whereas $\kappa_2 = \sigma_{\max}/\sigma_{\min}$ is a *ratio* with $\kappa(cA) = \kappa(A)$.

$$
\boxed{\det = \prod \sigma_i \text{ (scale)}, \qquad \kappa_2 = \sigma_{\max}/\sigma_{\min} \text{ (shape)} }
$$

*Key takeaway*: never test near-singularity with `det`; use `cond`, `svd`, or LAPACK's reciprocal condition estimate.

### Problem L0.4: The precision budget

A physical simulation must produce 4 correct decimal digits. The linear systems involved have $\kappa_2(A) \approx 10^{9}$. (a) Is binary32 sufficient? (b) Binary64? (c) How large may $\kappa$ grow before binary64 fails to deliver 4 digits?

**Solution**

Wilkinson's rule: correct digits $\approx p - \log_{10}\kappa$ where $u = 10^{-p}$.

**(a)** binary32: $p \approx 7.2$. Digits $\approx 7.2 - 9 = -1.8 \lt 0$: **not a single correct digit**. Binary32 is unusable here.

**(b)** binary64: $p \approx 15.95$. Digits $\approx 15.95 - 9 \approx 7$: comfortably above the requirement of 4, with 3 digits of margin for the algorithm's own constants.

**(c)** Require $15.95 - \log_{10}\kappa \ge 4$:

$$
\log_{10}\kappa \le 11.95 \quad \Longrightarrow \quad \kappa \lesssim 10^{12}
$$

$$
\boxed{\text{fp32: fails; fp64: } \approx 7 \text{ digits; budget exhausted at } \kappa \approx 10^{12}}
$$

*Key takeaway*: estimate $\log_{10}\kappa$ **before** choosing a dtype — precision selection is arithmetic on exponents, not a matter of taste.

## Level 1 — Foundation

### Problem L1.1: $\kappa_2$ of a nearly singular symmetric matrix, by hand

Let

$$
A = \begin{pmatrix} 1 & 1 \\ 1 & 1 + \delta \end{pmatrix}, \qquad 0 \lt \delta \ll 1
$$

Compute $\kappa_2(A)$ to leading order in $\delta$, and evaluate it for $\delta = 10^{-8}$.

**Solution**

$A$ is symmetric, so $\kappa_2 = \vert \lambda_{\max} \vert / \vert \lambda_{\min} \vert$. From the characteristic polynomial:

$$
\mathrm{tr}\,A = 2 + \delta, \qquad \det A = (1)(1+\delta) - 1 = \delta
$$

$$
\lambda_{\pm} = \frac{(2 + \delta) \pm \sqrt{(2+\delta)^{2} - 4\delta}}{2}
$$

Simplify the discriminant: $(2+\delta)^{2} - 4\delta = 4 + 4\delta + \delta^{2} - 4\delta = 4 + \delta^{2}$, so $\sqrt{\cdot} = 2\sqrt{1 + \delta^{2}/4} = 2 + \delta^{2}/4 + O(\delta^{4})$. Hence

$$
\lambda_{+} = 2 + \frac{\delta}{2} + O(\delta^{2}), \qquad \lambda_{-} = \frac{\delta}{2} - \frac{\delta^{2}}{8} + O(\delta^{3})
$$

(One can also read $\lambda_{-}$ straight off $\lambda_{+}\lambda_{-} = \det A = \delta$: $\lambda_{-} = \delta/\lambda_{+} \approx \delta/2$ — the numerically stable route, Vieta again.) Therefore

$$
\kappa_2(A) = \frac{\lambda_{+}}{\lambda_{-}} \approx \frac{2}{\delta/2} = \frac{4}{\delta}
$$

For $\delta = 10^{-8}$: $\kappa_2 \approx 4 \times 10^{8}$, so binary64 leaves $\approx 15.95 - 8.6 \approx 7$ correct digits.

$$
\boxed{\kappa_2(A) \approx \frac{4}{\delta}; \quad \delta = 10^{-8} \Rightarrow \kappa_2 \approx 4 \times 10^{8} \; (\approx 7 \text{ digits in fp64})}
$$

*Key takeaway*: computing $\lambda_{\min}$ as $\det/\lambda_{\max}$ instead of from the subtracting branch of the quadratic formula is Topic 02's stable-quadratic trick applied to a conditioning calculation.

### Problem L1.2: A small residual is not a small error

A solver returns $\hat{x}$ for $Ax = b$ with relative residual $\frac{\Vert b - A\hat{x} \Vert_2}{\Vert b \Vert_2} = 3 \times 10^{-16}$. Given $\kappa_2(A) = 4 \times 10^{9}$: (a) bound the relative forward error; (b) how many digits of $\hat{x}$ are trustworthy; (c) would a *different, better* algorithm help?

**Solution**

**(a)** By the corollary to Theorem 2.4,

$$
\frac{\Vert \hat{x} - x \Vert_2}{\Vert x \Vert_2} \le \kappa_2(A) \, \frac{\Vert r \Vert_2}{\Vert b \Vert_2} = 4 \times 10^{9} \times 3 \times 10^{-16} = 1.2 \times 10^{-6}
$$

**(b)** $\log_{10}(1/1.2 \times 10^{-6}) \approx 5.9$: about **6 correct digits**, even though the residual has 15.

**(c)** No. A residual of $3 \times 10^{-16}$ is already at the level of $O(u)$ — i.e. backward error $O(u)$, which is the *definition* of backward stability and cannot be improved within binary64. The remaining error is the problem's, not the algorithm's. The only levers are higher precision (iterative refinement in fp128, gaining $\sim 16$ more digits of residual), or reformulating/regularizing $A$.

$$
\boxed{\frac{\Vert \delta x \Vert}{\Vert x \Vert} \le 1.2 \times 10^{-6} \; (\approx 6 \text{ digits}); \text{ the algorithm is already optimal}}
$$

*Key takeaway*: residual certifies the algorithm; only $\kappa$ converts it to a statement about the answer. Report both.

### Problem L1.3: Algebra of condition numbers

Prove or refute, for nonsingular $A, B$ and orthogonal $Q$:
(a) $\kappa(cA) = \kappa(A)$ for $c \ne 0$;
(b) $\kappa(AB) \le \kappa(A)\kappa(B)$;
(c) $\kappa_2(QA) = \kappa_2(A)$;
(d) $\kappa_2(A^{k}) = \kappa_2(A)^{k}$;
(e) $\kappa(A) \ge 1$ in any submultiplicative norm with $\Vert I \Vert = 1$.

**Solution**

**(a) True.** $\Vert cA \Vert = \vert c \vert \Vert A \Vert$ and $\Vert (cA)^{-1} \Vert = \vert c \vert^{-1}\Vert A^{-1} \Vert$; the factors cancel.

**(b) True.** Submultiplicativity twice:

$$
\kappa(AB) = \Vert AB \Vert \, \Vert B^{-1}A^{-1} \Vert \le \Vert A \Vert \Vert B \Vert \Vert B^{-1} \Vert \Vert A^{-1} \Vert = \kappa(A)\kappa(B)
$$

**(c) True.** The 2-norm is orthogonally invariant: $QA$ has the same singular values as $A$, since $(QA)^{\top}(QA) = A^{\top}Q^{\top}QA = A^{\top}A$. This is *the* structural reason stable algorithms are built from Householder reflectors and Givens rotations: they reorganize a matrix without changing its conditioning.

**(d) Refuted in general, true for normal $A$.** For symmetric positive definite $A$ the eigenvalues are raised to the $k$-th power, giving $\kappa_2(A^{k}) = \kappa_2(A)^{k}$. In general only $\kappa_2(A^{k}) \le \kappa_2(A)^{k}$ holds. Counterexample: $A = \begin{pmatrix} 0 & 2 \\ 1/2 & 0 \end{pmatrix}$ has $\kappa_2(A) = 4$ but $A^{2} = I$, so $\kappa_2(A^{2}) = 1 \ne 16$.

**(e) True.** $1 = \Vert I \Vert = \Vert A A^{-1} \Vert \le \Vert A \Vert \Vert A^{-1} \Vert = \kappa(A)$.

$$
\boxed{\text{(a) T, (b) T, (c) T, (d) F in general (T for normal } A\text{), (e) T}}
$$

*Key takeaway*: orthogonal invariance (c) plus $\kappa \ge 1$ (e) is the entire design brief of dense numerical linear algebra — transform with orthogonals, because they are free.

### Problem L1.4: Hilbert matrices and the digit ledger

The Hilbert matrix $H_n$ has entries $h_{ij} = 1/(i + j - 1)$, with $\kappa_2(H_3) \approx 5.2 \times 10^{2}$, $\kappa_2(H_6) \approx 1.5 \times 10^{7}$, $\kappa_2(H_{10}) \approx 1.6 \times 10^{13}$, $\kappa_2(H_{12}) \approx 1.6 \times 10^{16}$. (a) How many digits of $x$ survive a binary64 solve of $H_n x = b$ for each $n$? (b) At which $n$ does binary64 fail completely? (c) Where do Hilbert matrices come from, and what is the fix?

**Solution**

**(a)** Digits $\approx 15.95 - \log_{10}\kappa_2$:

| $n$ | $\kappa_2(H_n)$ | $\log_{10}\kappa_2$ | fp64 digits | fp32 digits |
|---|---|---|---|---|
| 3 | $5.2 \times 10^{2}$ | 2.7 | 13 | 4.5 |
| 6 | $1.5 \times 10^{7}$ | 7.2 | 8.8 | 0 |
| 10 | $1.6 \times 10^{13}$ | 13.2 | 2.8 | 0 |
| 12 | $1.6 \times 10^{16}$ | 16.2 | 0 | 0 |

**(b)** At $n = 12$: $\kappa_2 u \approx 1.6 \times 10^{16} \times 1.1 \times 10^{-16} \approx 1.8 \gt 1$, so the *relative* error bound exceeds 100%. The computed solution may share no digits with the truth — and indeed the matrix is numerically singular, since $1/\kappa_2 \lt u$ means $H_{12}$ is within rounding distance of a singular matrix (Theorem 2.4b).

**(c)** $H_n$ is the Gram matrix of the monomials $1, t, t^{2}, \dots$ on $[0, 1]$: $\int_0^{1} t^{i-1}t^{j-1}dt = 1/(i+j-1)$. So *every least-squares polynomial fit in the monomial basis via normal equations* builds a Hilbert matrix. The fix is a change of basis to an orthogonal family (Legendre on $[-1,1]$, Chebyshev for interpolation), for which the Gram matrix is diagonal and $\kappa = 1$, combined with QR instead of normal equations. Growth is roughly $\kappa_2(H_n) \sim e^{3.5n}$ — exponential, so no precision saves the monomial route beyond small $n$.

$$
\boxed{\text{fp64 digits} \approx 16 - \log_{10}\kappa_2(H_n); \; n = 12 \text{ is numerically singular; fix = orthogonal basis} + \text{QR}}
$$

*Key takeaway*: an ill-conditioned matrix is usually a symptom of a bad *basis*, and changing basis is free compared with fighting the conditioning.

### Problem L1.5: Condition number of the matrix–vector product

For fixed nonsingular $A$, consider the map $x \mapsto Ax$. (a) Derive its relative condition number at $x$. (b) Show it is bounded by $\kappa(A)$ and give the vectors attaining the extremes. (c) Why is $\kappa(A)$ still the right number to quote for the *solve*?

**Solution**

**(a)** The map is linear, so $J = A$ and by Theorem 2.2

$$
\kappa_{A}(x) = \frac{\Vert A \Vert \, \Vert x \Vert}{\Vert Ax \Vert}
$$

**(b)** Since $\Vert x \Vert = \Vert A^{-1}Ax \Vert \le \Vert A^{-1} \Vert \Vert Ax \Vert$,

$$
\kappa_A(x) \le \Vert A \Vert \Vert A^{-1} \Vert = \kappa(A)
$$

In the 2-norm with SVD $A = U\Sigma V^{\top}$: taking $x = v_n$ (right singular vector of $\sigma_{\min}$) gives $\Vert Ax \Vert_2 = \sigma_{\min}$ and $\kappa_A = \sigma_{\max}/\sigma_{\min} = \kappa_2(A)$ — the maximum. Taking $x = v_1$ gives $\Vert Ax \Vert_2 = \sigma_{\max}$ and $\kappa_A = 1$ — the minimum. So multiplication by an ill-conditioned matrix is dangerous **only for inputs aligned with its weak directions**.

**(c)** Solving $Ax = b$ is applying $A^{-1}$ to $b$; its pointwise condition number is $\Vert A^{-1} \Vert \Vert b \Vert / \Vert A^{-1}b \Vert \le \kappa(A)$. Worst-case over $b$ (which is what a general-purpose solver must guarantee) is exactly $\kappa(A)$, and it is attained — hence the convention. For a *specific* right-hand side aligned with the dominant singular direction the effective conditioning can be far better, which is why some ill-conditioned systems in practice still give good answers.

$$
\boxed{\kappa_A(x) = \frac{\Vert A \Vert \Vert x \Vert}{\Vert Ax \Vert} \le \kappa(A), \text{ attained at } x = v_{n}, \text{ minimal at } x = v_{1}}
$$

*Key takeaway*: $\kappa(A)$ is a worst-case over right-hand sides. When you know $b$, the *effective* condition number may be much smaller — measure it before panicking.

### Problem L1.6: Distance to singularity

Let $A$ be $500 \times 500$ with $\Vert A \Vert_2 = 1$ and $\kappa_2(A) = 10^{12}$, stored in binary64. (a) How far (in relative 2-norm) is $A$ from the nearest singular matrix? (b) Is that distance resolvable in binary64? (c) In binary32? Interpret.

**Solution**

**(a)** By Theorem 2.4b (Kahan/Gastinel),

$$
\frac{\mathrm{dist}_2(A, \text{singular})}{\Vert A \Vert_2} = \frac{1}{\kappa_2(A)} = 10^{-12}
$$

so with $\Vert A \Vert_2 = 1$ the nearest singular matrix is at absolute distance $\sigma_{\min} = 10^{-12}$.

**(b)** Rounding $A$ into binary64 perturbs it by $\Vert \Delta A \Vert_2 \lesssim \sqrt{n}\,u\Vert A \Vert_2 \approx 22 \times 1.1 \times 10^{-16} \approx 2.4 \times 10^{-15}$. Since $2.4 \times 10^{-15} \ll 10^{-12}$, the stored matrix is *provably nonsingular*: rounding cannot reach the singular set. Solving is meaningful, with about $16 - 12 = 4$ correct digits.

**(c)** In binary32, $\Vert \Delta A \Vert_2 \approx 22 \times 6 \times 10^{-8} \approx 1.3 \times 10^{-6} \gg 10^{-12}$. The rounded matrix is indistinguishable from singular ones: $A$ is *numerically singular in fp32*, and no fp32 solve carries information.

$$
\boxed{\mathrm{dist} = \Vert A \Vert_2/\kappa_2 = 10^{-12}: \text{ nonsingular in fp64 } (\Delta \sim 10^{-15}), \text{ singular in fp32 } (\Delta \sim 10^{-6})}
$$

*Key takeaway*: "numerically singular" is precision-relative — it means $1/\kappa \lesssim u$, i.e. rounding alone can reach a singular matrix.

## Level 2 — Applications in AI/ML

### Problem L2.1: Polynomial regression — normal equations versus QR

You fit a degree-9 polynomial in the raw monomial basis to data on $[0, 1]$, with design matrix $X \in \mathbb{R}^{1000 \times 10}$ having $\kappa_2(X) \approx 3 \times 10^{6}$. (a) Predict the accuracy of $\hat{\beta}$ via normal equations ($X^{\top}X$ + Cholesky) in fp64 and fp32. (b) Same for `numpy.linalg.lstsq`. (c) What is the practitioner's fix?

**Solution**

**(a)** Normal equations solve a system with

$$
\kappa_2(X^{\top}X) = \kappa_2(X)^{2} = (3 \times 10^{6})^{2} = 9 \times 10^{12}
$$

- fp64: digits $\approx 15.95 - 12.95 \approx 3$. Barely usable; coefficient signs may be right but little else.
- fp32: digits $\approx 7.2 - 12.95 \lt 0$. **Complete failure** — and Cholesky will typically abort with "matrix not positive definite", because the computed Gram matrix has a negative eigenvalue.

**(b)** QR/SVD-based `lstsq` works at the intrinsic level $\kappa_2(X)$ (small residual assumed):

- fp64: digits $\approx 15.95 - 6.5 \approx 9.5$.
- fp32: digits $\approx 7.2 - 6.5 \approx 0.7$ — marginal, but not catastrophic.

The gap between (a) and (b) in fp64 is **six digits, for free**, purely from not forming $X^{\top}X$.

**(c)** Two moves, in order:
1. **Change basis**: fit in a Chebyshev or Legendre basis (or equivalently center/scale $t$ to $[-1, 1]$ and orthogonalize). This drops $\kappa_2(X)$ from $10^{6}$ to $O(10)$ — a far bigger win than any solver choice.
2. **Use `lstsq` / QR**, never `inv(X.T @ X) @ X.T @ y`.

```python
# numpy.polynomial.Polynomial.fit uses a scaled/shifted domain and lstsq internally:
# from numpy.polynomial import Polynomial
# p = Polynomial.fit(t, y, deg=9)     # well-conditioned by construction
```

$$
\boxed{\text{normal eq.: } \kappa^{2} = 9 \times 10^{12} \Rightarrow \sim 3 \text{ fp64 digits}; \quad \text{QR: } \kappa = 3 \times 10^{6} \Rightarrow \sim 9.5 \text{ digits}}
$$

*Key takeaway*: `lstsq` is not a convenience wrapper around the normal equations — it is a different, quadratically better-conditioned algorithm.

### Problem L2.2: Choosing the ridge parameter as a conditioning cap

A design matrix has $\sigma_{\max}(X) = 50$ and $\sigma_{\min}(X) = 5 \times 10^{-7}$. (a) Compute $\kappa_2(X)$ and $\kappa_2(X^{\top}X)$. (b) Find the smallest $\lambda$ making $\kappa_2(X^{\top}X + \lambda I) \le 10^{8}$. (c) What fraction of the signal does that $\lambda$ suppress, in terms of SVD filter factors?

**Solution**

**(a)**

$$
\kappa_2(X) = \frac{50}{5 \times 10^{-7}} = 10^{8}, \qquad \kappa_2(X^{\top}X) = 10^{16}
$$

The Gram matrix is numerically singular in binary64.

**(b)** From Derivation 3.6,

$$
\kappa_2(X^{\top}X + \lambda I) = \frac{\sigma_{\max}^{2} + \lambda}{\sigma_{\min}^{2} + \lambda} \le 10^{8}
$$

With $\sigma_{\max}^{2} = 2500$ and $\sigma_{\min}^{2} = 2.5 \times 10^{-13}$ (negligible against any useful $\lambda$), the condition simplifies to $\frac{2500 + \lambda}{\lambda} \le 10^{8}$, i.e.

$$
\lambda \ge \frac{2500}{10^{8} - 1} \approx 2.5 \times 10^{-5}
$$

**(c)** The filter factor is $f_i = \sigma_i^{2}/(\sigma_i^{2} + \lambda)$ with $\lambda = 2.5 \times 10^{-5}$, so the half-power point sits at $\sigma_i = \sqrt{\lambda} = 5 \times 10^{-3}$:

- Directions with $\sigma_i \gg 5 \times 10^{-3}$: $f_i \approx 1$, untouched.
- Direction with $\sigma_{\min} = 5 \times 10^{-7}$: $f = \frac{2.5 \times 10^{-13}}{2.5 \times 10^{-13} + 2.5 \times 10^{-5}} \approx 10^{-8}$ — annihilated.

So $\lambda$ deletes precisely the directions whose coefficients would have been $1/\sigma_i \sim 2 \times 10^{6}$ times amplified noise, and leaves everything the data actually determines.

$$
\boxed{\lambda \ge \sigma_{\max}^{2}/\kappa_{\text{target}} \approx 2.5 \times 10^{-5}; \; \text{cutoff at } \sigma = \sqrt{\lambda} = 5 \times 10^{-3}}
$$

*Key takeaway*: the general recipe $\lambda \approx \sigma_{\max}^{2}/\kappa_{\text{target}}$ turns "how much regularization?" into "how many digits do I want to keep?".

### Problem L2.3: Gaussian-process jitter

An RBF-kernel Gram matrix $K \in \mathbb{R}^{500 \times 500}$ has $\lambda_{\max} = 480$ and eigenvalues decaying to $\lambda_{\min} \approx 10^{-18}$. Cholesky in fp64 fails. (a) Explain why, using conditioning. (b) Choose a jitter $\varepsilon$ giving $\kappa_2 \le 10^{10}$. (c) Give the statistical interpretation of $\varepsilon$ and one alternative fix.

**Solution**

**(a)** $\kappa_2(K) = 480/10^{-18} \approx 5 \times 10^{20} \gg 1/u \approx 9 \times 10^{15}$, so $K$ is numerically singular: rounding the entries alone can push the smallest eigenvalue below zero, and Cholesky — which must take $\sqrt{\cdot}$ of pivots — aborts. Mathematically $K \succ 0$ (an RBF kernel matrix on distinct points is positive definite); the failure is entirely a conditioning artifact of representing $K$ in binary64.

**(b)** Adding $\varepsilon I$ shifts every eigenvalue up:

$$
\kappa_2(K + \varepsilon I) = \frac{\lambda_{\max} + \varepsilon}{\lambda_{\min} + \varepsilon} \approx \frac{480}{\varepsilon} \le 10^{10} \quad \Longrightarrow \quad \varepsilon \ge 4.8 \times 10^{-8}
$$

A practical choice is $\varepsilon = 10^{-6}\,\lambda_{\max} \approx 5 \times 10^{-4}$, giving $\kappa_2 \approx 10^{6}$ with ample margin. Common libraries start at $10^{-6}$ relative and escalate by factors of 10 until Cholesky succeeds.

**(c)** Statistically, $K + \varepsilon I$ is the covariance of the *same* GP observed with independent Gaussian noise of variance $\varepsilon$ — so jitter is not a hack but a (very small) noise floor, identical in form to the likelihood's $\sigma_n^{2}I$. Alternatives: (i) fold jitter into the learned noise variance and let the marginal likelihood choose it; (ii) use inducing-point / low-rank approximations (SVGP, Nyström) that never form the full $K$; (iii) use a less smooth kernel (Matérn $\nu = 3/2$) whose eigenvalues decay polynomially rather than exponentially, so $\kappa_2$ grows far more slowly.

$$
\boxed{\varepsilon \ge \lambda_{\max}/\kappa_{\text{target}} \approx 5 \times 10^{-8}; \text{ jitter } = \text{ ridge on the kernel } = \text{ an observation-noise floor}}
$$

*Key takeaway*: "add jitter until Cholesky works" is Derivation 3.6 executed by trial and error — computing $\lambda_{\max}/\kappa_{\text{target}}$ gets it right on the first attempt.

### Problem L2.4: Hessian conditioning sets the iteration count

A quadratic model of a training loss has Hessian eigenvalues spanning $\lambda_{\max} = 200$, $\lambda_{\min} = 0.02$. (a) Compute $\kappa$. (b) How many gradient-descent steps (optimal fixed step size) reduce the error by $10^{-3}$? (c) With Nesterov momentum? (d) What does a diagonal preconditioner buy if it equalizes the diagonal?

**Solution**

**(a)** $\kappa = 200/0.02 = 10^{4}$.

**(b)** The optimal-step contraction factor is

$$
\rho = \frac{\kappa - 1}{\kappa + 1} = 1 - \frac{2}{\kappa + 1} \approx 1 - 2 \times 10^{-4}
$$

Requiring $\rho^{N} \le 10^{-3}$:

$$
N \ge \frac{\ln 10^{3}}{-\ln \rho} \approx \frac{6.91}{2 \times 10^{-4}} \approx 3.5 \times 10^{4} \text{ iterations}
$$

**(c)** Nesterov/heavy-ball achieve $\rho_{\text{acc}} \approx 1 - 2/\sqrt{\kappa} = 1 - 0.02$:

$$
N \ge \frac{6.91}{0.02} \approx 3.5 \times 10^{2} \text{ iterations}
$$

a **100-fold** reduction, exactly the factor $\sqrt{\kappa} = 100$.

**(d)** A diagonal preconditioner $D^{-1/2}HD^{-1/2}$ with $D = \mathrm{diag}(H)$ removes conditioning caused by *scale mismatch across coordinates* (differing units, differing feature variances) but not conditioning caused by *correlation* (off-diagonal structure). Van der Sluis's theorem bounds the residual conditioning: for symmetric positive definite $H$ with unit-diagonal scaling, $\kappa(D^{-1/2}HD^{-1/2}) \le n \min_{D'} \kappa(D'HD')$. In practice this is why Adam (a diagonal preconditioner) helps enormously on badly scaled parameterizations and rather little on rotationally ill-conditioned losses, where full-matrix methods (K-FAC, Shampoo) are needed.

$$
\boxed{\kappa = 10^{4}: \; N_{\text{GD}} \approx 3.5 \times 10^{4}, \quad N_{\text{Nesterov}} \approx 3.5 \times 10^{2} \; (\text{speedup } \sqrt{\kappa} = 100)}
$$

*Key takeaway*: in optimization $\kappa$ buys *iterations*, in linear algebra it costs *digits* — same quantity, two currencies.

### Problem L2.5: Collinear features — coefficients versus predictions

Features $x_1$ and $x_2 = x_1 + \epsilon z$ with $\epsilon = 10^{-6}$, $z$ orthonormal to $x_1$, $\Vert x_1 \Vert_2 = 1$, $n$ large. The response is $y = x_1 + \text{noise}$. (a) Estimate $\kappa_2(X)$ for $X = [x_1 \; x_2]$. (b) Explain the classical "coefficients flip sign between refits" symptom quantitatively. (c) Why do predictions stay accurate? (d) Give the fix.

**Solution**

**(a)** In the orthonormal basis $\{x_1, z\}$ the design matrix is $\begin{pmatrix} 1 & 1 \\ 0 & \epsilon \end{pmatrix}$. Its singular values satisfy $\sigma_{\max} \approx \sqrt{2}$ and $\sigma_{\max}\sigma_{\min} = \vert \det \vert = \epsilon$, so $\sigma_{\min} \approx \epsilon/\sqrt{2}$ and

$$
\kappa_2(X) \approx \frac{\sqrt{2}}{\epsilon/\sqrt{2}} = \frac{2}{\epsilon} = 2 \times 10^{6}
$$

**(b)** Theorem 2.4 says a relative data perturbation $\eta$ (statistical noise, a different train/test split, a change of random seed) moves $\hat{\beta}$ by up to $\kappa_2(X)\eta = 2 \times 10^{6}\eta$. With $\eta = 10^{-3}$ noise, $\hat{\beta}$ can move by $2 \times 10^{3}$ — far larger than the coefficients themselves, so signs flip freely. Concretely $\beta = (1, 0)$ and $\beta' = (1 + t, -t)\cdot$(approximately) give nearly identical fits for any $t$, because $x_1 - x_2 = -\epsilon z$ is nearly zero.

**(c)** Predictions are $X\hat{\beta}$, i.e. the *forward* map, whose condition number is $\Vert X \Vert \Vert \beta \Vert / \Vert X\beta \Vert$ — small whenever $\hat{\beta}$ has little energy in the weak singular direction. The huge coefficient swings live almost entirely in the near-null direction $v_2$, where $X v_2 \approx 0$: they cancel on prediction. This is the precise sense in which multicollinearity harms *interpretation* but not *prediction*.

**(d)** Ridge with $\lambda \gtrsim \sigma_{\max}^{2}/\kappa_{\text{target}}$ (Problem L2.2), or drop/merge the duplicate feature, or report the identified combination $\beta_1 + \beta_2$ instead of the individual coefficients.

$$
\boxed{\kappa_2(X) \approx 2/\epsilon = 2 \times 10^{6}: \text{ coefficients unidentified along } v_2, \text{ predictions unaffected}}
$$

*Key takeaway*: ask "conditioning of which map?" — $y \mapsto \hat{\beta}$ can be hopeless while $y \mapsto X\hat{\beta}$ is perfect.

### Problem L2.6: Feature scaling is a conditioning decision

A regression uses two features: age in years (range $20$–$70$) and income in dollars (range $2 \times 10^{4}$–$2 \times 10^{5}$). (a) Estimate $\kappa_2(X^{\top}X)$ from the scale mismatch alone, assuming uncorrelated features. (b) What does standardizing do? (c) Connect to gradient descent and to van der Sluis's theorem.

**Solution**

**(a)** For uncorrelated, mean-removed columns, $X^{\top}X \approx n\,\mathrm{diag}(s_1^{2}, s_2^{2})$ with standard deviations $s_{\text{age}} \approx 15$ and $s_{\text{income}} \approx 5 \times 10^{4}$. Hence

$$
\kappa_2(X^{\top}X) \approx \left( \frac{5 \times 10^{4}}{15} \right)^{2} \approx (3.3 \times 10^{3})^{2} \approx 1.1 \times 10^{7}
$$

Seven orders of magnitude of conditioning, generated entirely by the choice of *units* — nothing statistical about it. Switching income to thousands of dollars would remove three of them.

**(b)** Standardizing ($x \mapsto (x - \mu)/s$) makes both columns unit variance, so the diagonal is equalized and

$$
\kappa_2(X^{\top}X) \approx \frac{1 + \rho}{1 - \rho}
$$

for correlation $\rho$ — now determined by genuine statistical dependence rather than units. With $\rho = 0.5$, $\kappa_2 = 3$.

**(c)** Gradient descent on the unstandardized problem has $\kappa = 1.1 \times 10^{7}$ and needs $O(\kappa)$ iterations (Problem L2.4) — millions — while the standardized problem converges in a handful. This is why standardization is mandatory for gradient-based linear models, SVMs, k-NN, and neural nets, and irrelevant for scale-invariant learners such as decision trees. Van der Sluis's theorem justifies the specific choice: equilibrating rows/columns to unit norm is within a factor $n$ of the *optimal* diagonal scaling, so no cleverer rescaling can help much more.

$$
\boxed{\text{unstandardized } \kappa_2 \approx 10^{7} \text{ (pure units)} \; \longrightarrow \; \text{standardized } \kappa_2 = \frac{1+\rho}{1-\rho} \approx 3}
$$

*Key takeaway*: `StandardScaler` is a preconditioner. It buys digits in the solve and iterations in the optimizer, at zero statistical cost for linear models.

## Level 3 — Challenge

### Problem L3.1: The squaring theorem and its sharpness

(a) Prove $\kappa_2(X^{\top}X) = \kappa_2(X)^{2}$ for full-column-rank $X$. (b) Exhibit the Läuchli matrix and the exact threshold at which the fp64 Gram matrix becomes singular. (c) Conclude quantitatively how much precision the normal equations waste.

**Solution**

**(a)** Let $X = U\Sigma V^{\top}$ be the thin SVD with $U^{\top}U = I_n$, $V$ orthogonal, $\Sigma = \mathrm{diag}(\sigma_1 \ge \cdots \ge \sigma_n \gt 0)$. Then

$$
X^{\top}X = V\Sigma U^{\top}U\Sigma V^{\top} = V\Sigma^{2}V^{\top}
$$

a symmetric eigendecomposition with eigenvalues $\sigma_i^{2} \gt 0$. Since for symmetric positive definite matrices $\kappa_2 = \lambda_{\max}/\lambda_{\min}$,

$$
\kappa_2(X^{\top}X) = \frac{\sigma_1^{2}}{\sigma_n^{2}} = \kappa_2(X)^{2} \qquad \blacksquare
$$

**(b)** Läuchli:

$$
X = \begin{pmatrix} 1 & 1 \\ \epsilon & 0 \\ 0 & \epsilon \end{pmatrix}, \qquad X^{\top}X = \begin{pmatrix} 1 + \epsilon^{2} & 1 \\ 1 & 1 + \epsilon^{2} \end{pmatrix}
$$

Exact eigenvalues of the Gram matrix: $2 + \epsilon^{2}$ and $\epsilon^{2}$, so $\kappa_2(X^{\top}X) = (2 + \epsilon^{2})/\epsilon^{2} \approx 2/\epsilon^{2}$ and $\kappa_2(X) \approx \sqrt{2}/\epsilon$ — the squaring, verified. In floating point, $\mathrm{fl}(1 + \epsilon^{2}) = 1$ as soon as

$$
\epsilon^{2} \lt u \quad \Longleftrightarrow \quad \epsilon \lt \sqrt{u} \approx 1.05 \times 10^{-8} \; (\text{binary64})
$$

At that point the computed Gram matrix is $\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$, exactly singular: Cholesky fails, and the information distinguishing the two columns has been destroyed by a *single rounding*. Meanwhile $X$ itself has $\kappa_2 \approx 1.3 \times 10^{8}$ — a QR-based solve returns 7–8 correct digits.

**(c)** The threshold $\epsilon = \sqrt{u}$ is the general statement: **forming a Gram matrix costs exactly half of the available precision.** In binary64 the normal equations are safe only while $\kappa_2(X) \lesssim u^{-1/2} \approx 10^{8}$; QR remains safe to $\kappa_2(X) \approx 10^{16}$.

$$
\boxed{\kappa_2(X^{\top}X) = \kappa_2(X)^{2}; \text{ normal equations lose half the digits, failing at } \kappa_2(X) \approx u^{-1/2}}
$$

*Key takeaway*: the "half precision" rule generalizes — every squaring of the data (Gram matrices, covariance from raw second moments, $\Vert \cdot \Vert^{2}$ objectives) halves the exponent budget.

### Problem L3.2: Prove the distance-to-singularity theorem

Prove that for nonsingular $A$,

$$
\min \left\{ \Vert \Delta A \Vert_2 \; : \; A + \Delta A \text{ singular} \right\} = \sigma_{\min}(A)
$$

and deduce $\mathrm{dist}_2(A, \text{singular})/\Vert A \Vert_2 = 1/\kappa_2(A)$.

**Solution**

Let $A = U\Sigma V^{\top}$ with $\sigma_n = \sigma_{\min}$, and let $u_n, v_n$ be the corresponding singular vectors.

**Step 1 — attainability ($\le$).** Take $\Delta A = -\sigma_n u_n v_n^{\top}$, a rank-one matrix with $\Vert \Delta A \Vert_2 = \sigma_n$ (the 2-norm of a rank-one $uv^{\top}$ with unit vectors is $1$). Then

$$
(A + \Delta A) v_n = A v_n - \sigma_n u_n (v_n^{\top}v_n) = \sigma_n u_n - \sigma_n u_n = 0
$$

so $A + \Delta A$ has $v_n$ in its null space and is singular. Hence the minimum is at most $\sigma_n$.

**Step 2 — lower bound ($\ge$).** Suppose $A + \Delta A$ is singular: there exists $w$ with $\Vert w \Vert_2 = 1$ and $(A + \Delta A)w = 0$, i.e. $Aw = -\Delta A\, w$. Taking norms and using the variational characterization $\Vert Aw \Vert_2 \ge \sigma_{\min}\Vert w \Vert_2$ (immediate from $\Vert Aw \Vert_2^{2} = \sum_i \sigma_i^{2}(v_i^{\top}w)^{2} \ge \sigma_n^{2}\Vert w \Vert_2^{2}$):

$$
\sigma_n = \sigma_n \Vert w \Vert_2 \le \Vert A w \Vert_2 = \Vert \Delta A\, w \Vert_2 \le \Vert \Delta A \Vert_2 \Vert w \Vert_2 = \Vert \Delta A \Vert_2
$$

So every singularizing perturbation has norm at least $\sigma_n$. Combining the two steps gives equality. $\blacksquare$

**Deduction.** Divide by $\Vert A \Vert_2 = \sigma_{\max}$:

$$
\frac{\mathrm{dist}_2(A, \text{singular})}{\Vert A \Vert_2} = \frac{\sigma_{\min}}{\sigma_{\max}} = \frac{1}{\kappa_2(A)}
$$

**Two consequences worth stating.**

1. **"Numerically singular" has a definition**: $A$ is numerically singular at precision $u$ when $1/\kappa_2(A) \lesssim u$, i.e. when representing $A$ in that precision can already reach the singular set.
2. This is the $r = 0$ case of the **Eckart–Young–Mirsky theorem**: the best rank-$k$ approximation error in the 2-norm is $\sigma_{k+1}$. Distance to singularity, truncated SVD, and PCA's reconstruction error are all one statement.

$$
\boxed{\mathrm{dist}_2(A, \text{singular}) = \sigma_{\min}(A) = \frac{\Vert A \Vert_2}{\kappa_2(A)}}
$$

*Key takeaway*: $1/\kappa_2$ is a *distance*, which is why condition numbers behave like inverse safety margins rather than like error estimates.

### Problem L3.3: Eigenvalues of a near-defective matrix

Consider the $k \times k$ Jordan-type matrix $J_k(\mu)$ with $\mu$ on the diagonal and $1$ on the superdiagonal, perturbed in the bottom-left corner:

$$
A(\varepsilon) = J_k(\mu) + \varepsilon\, e_k e_1^{\top}
$$

(a) Compute the eigenvalues exactly. (b) Deduce the perturbation exponent and evaluate for $k = 4$, $\varepsilon = 10^{-16}$. (c) Reconcile with Theorem 2.6, and state what this means for eigenvalue computations on non-normal matrices.

**Solution**

**(a)** Expanding the determinant of $A(\varepsilon) - \lambda I$ along the first column (the only nonzero entries are $\mu - \lambda$ at the top and $\varepsilon$ at the bottom) gives the characteristic polynomial

$$
\det\!\left( A(\varepsilon) - \lambda I \right) = (\mu - \lambda)^{k} + (-1)^{k+1}\varepsilon \cdot (-1)^{k+1} = (\mu - \lambda)^{k} - (-1)^{k}\varepsilon
$$

so, up to the sign convention, $(\lambda - \mu)^{k} = \varepsilon$ and the eigenvalues are the $k$ points

$$
\lambda_j = \mu + \varepsilon^{1/k}\, e^{2\pi i j/k}, \qquad j = 0, 1, \dots, k-1
$$

— a regular $k$-gon of radius $\varepsilon^{1/k}$ centred at $\mu$.

**(b)** The displacement is $\varepsilon^{1/k}$, **not** $O(\varepsilon)$. For $k = 4$, $\varepsilon = 10^{-16}$:

$$
\vert \lambda_j - \mu \vert = (10^{-16})^{1/4} = 10^{-4}
$$

A perturbation at the level of one binary64 rounding moves the eigenvalues by $10^{-4}$ — twelve orders of magnitude of amplification. The absolute condition number in the usual sense is *infinite*, since $\lim_{\varepsilon \to 0}\varepsilon^{1/k}/\varepsilon = \infty$.

**(c)** Theorem 2.6 assumed $\lambda$ **simple**. Here $\mu$ has algebraic multiplicity $k$ but geometric multiplicity $1$: the matrix is defective, left and right eigenvectors are orthogonal ($y^{*}x = e_k^{\top}e_1 = 0$), so $\hat{\kappa}_\lambda = 1/\vert y^{*}x \vert = \infty$ — the theorem's formula correctly predicts its own breakdown. Practical consequences:

- `numpy.linalg.eig` on a non-normal matrix can return eigenvalues wrong in the leading digits while being *backward stable*: it returns the exact eigenvalues of $A + \Delta A$ with $\Vert \Delta A \Vert = O(u\Vert A \Vert)$, which for a defective matrix is all one can ask.
- Always prefer `eigh` when the matrix is symmetric — it exploits $\hat{\kappa}_\lambda = 1$ and is accurate to $O(u\Vert A \Vert)$ absolutely.
- For non-normal operators the physically meaningful object is the **pseudospectrum** $\Lambda_\epsilon(A) = \{ z : \sigma_{\min}(zI - A) \le \epsilon \}$, which predicts the transient amplification (fluid instability, non-normal RNN dynamics) that eigenvalues alone miss.

$$
\boxed{\vert \Delta\lambda \vert = \varepsilon^{1/k}: \; k = 4, \varepsilon = 10^{-16} \Rightarrow 10^{-4}; \; \hat{\kappa}_\lambda = 1/\vert y^{*}x \vert = \infty \text{ (defective)}}
$$

*Key takeaway*: non-normality, not size or determinant, is what makes eigenvalue problems dangerous — and the diagnostic is the left/right eigenvector angle.

### Problem L3.4: Ridge filter factors — the optimal conditioning/bias trade-off

Let $X = U\Sigma V^{\top}$ (thin SVD), $y = X\beta^{*} + \eta$ with $\eta$ zero-mean, isotropic, variance $s^{2}$. Write everything in the SVD basis with $b_i = v_i^{\top}\beta^{*}$.

(a) Derive $\hat{\beta}_\lambda$ componentwise and identify the filter factors. (b) Derive bias and variance per component. (c) Minimize the total mean-squared error over $\lambda$ and interpret the result as a conditioning statement.

**Solution**

**(a)** From $(X^{\top}X + \lambda I)\hat{\beta}_\lambda = X^{\top}y$ and $X^{\top}X = V\Sigma^{2}V^{\top}$, $X^{\top} = V\Sigma U^{\top}$:

$$
\hat{\beta}_\lambda = V(\Sigma^{2} + \lambda I)^{-1}\Sigma U^{\top} y \quad \Longrightarrow \quad v_i^{\top}\hat{\beta}_\lambda = \frac{\sigma_i}{\sigma_i^{2} + \lambda}\, u_i^{\top}y = f_i \cdot \frac{u_i^{\top}y}{\sigma_i}
$$

with the **filter factors**

$$
f_i = \frac{\sigma_i^{2}}{\sigma_i^{2} + \lambda} \in (0, 1)
$$

At $\lambda = 0$ this is ordinary least squares, $v_i^{\top}\hat{\beta} = u_i^{\top}y/\sigma_i$ — note the explicit $1/\sigma_i$ amplification of noise in weak directions, which *is* the ill-conditioning.

**(b)** Substituting $u_i^{\top}y = \sigma_i b_i + u_i^{\top}\eta$ with $\mathrm{Var}(u_i^{\top}\eta) = s^{2}$:

$$
\mathbb{E}\!\left[ v_i^{\top}\hat{\beta}_\lambda \right] = f_i b_i \quad \Longrightarrow \quad \text{bias}_i = -(1 - f_i) b_i = -\frac{\lambda}{\sigma_i^{2} + \lambda}\, b_i
$$

$$
\mathrm{Var}_i = \left( \frac{\sigma_i}{\sigma_i^{2} + \lambda} \right)^{2} s^{2} = \frac{f_i^{2}}{\sigma_i^{2}}\, s^{2}
$$

At $\lambda = 0$: zero bias, variance $s^{2}/\sigma_i^{2}$ — unbounded as $\sigma_i \to 0$. As $\lambda \to \infty$: zero variance, bias $= -b_i$ (the estimate collapses to zero).

**(c)** Total MSE:

$$
\mathrm{MSE}(\lambda) = \sum_i \left[ \frac{\lambda^{2} b_i^{2}}{(\sigma_i^{2} + \lambda)^{2}} + \frac{\sigma_i^{2} s^{2}}{(\sigma_i^{2} + \lambda)^{2}} \right] = \sum_i \frac{\lambda^{2}b_i^{2} + \sigma_i^{2}s^{2}}{(\sigma_i^{2} + \lambda)^{2}}
$$

Differentiate a single term with respect to $\lambda$:

$$
\frac{d}{d\lambda} \frac{\lambda^{2}b_i^{2} + \sigma_i^{2}s^{2}}{(\sigma_i^{2} + \lambda)^{2}} = \frac{2\lambda b_i^{2}(\sigma_i^{2}+\lambda)^{2} - 2(\sigma_i^{2}+\lambda)(\lambda^{2}b_i^{2} + \sigma_i^{2}s^{2})}{(\sigma_i^{2}+\lambda)^{4}} = \frac{2\sigma_i^{2}\left( \lambda b_i^{2} - s^{2} \right)}{(\sigma_i^{2}+\lambda)^{3}}
$$

Each term is minimized at $\lambda = s^{2}/b_i^{2}$; if the signal is isotropic, $b_i^{2} \approx \Vert \beta^{*} \Vert^{2}/n$, all terms agree and

$$
\lambda^{*} = \frac{s^{2}}{\mathbb{E}[b_i^{2}]} = \frac{n\,s^{2}}{\Vert \beta^{*} \Vert_2^{2}}
$$

— the classical noise-to-signal ratio, which is also the Bayesian MAP answer under $\beta \sim \mathcal{N}(0, \tau^{2}I)$ with $\lambda = s^{2}/\tau^{2}$.

**Conditioning reading.** $\lambda^{*}$ places the filter cutoff at $\sigma_i = \sqrt{\lambda^{*}} = s/\sqrt{\mathbb{E}[b_i^{2}]}$: directions whose signal $\sigma_i b_i$ exceeds the noise $s$ are kept, the rest discarded. Statistical optimality and numerical conditioning agree — **the directions worth keeping are exactly the well-conditioned ones**, and the resulting condition number is capped at $\kappa_2 \le 1 + \sigma_{\max}^{2}/\lambda^{*}$.

$$
\boxed{f_i = \frac{\sigma_i^{2}}{\sigma_i^{2} + \lambda}, \quad \lambda^{*} = \frac{s^{2}}{\mathbb{E}[b_i^{2}]}, \quad \text{cutoff at } \sigma_i = \sqrt{\lambda^{*}}, \quad \kappa_2 \le 1 + \frac{\sigma_{\max}^{2}}{\lambda^{*}}}
$$

*Key takeaway*: regularization is not a numerical patch bolted onto statistics — the signal-to-noise cutoff and the condition-number cap are the same threshold, derived twice.